In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# Notebook: PASSO 4 — PCA (Algoritmo 2 do artigo) com retenção de 95% da variância
# Entrada:  emodnet_chemistry_integrated (já com colunas *_z + WQI + QualityClass)
# Saídas:
#   1) emodnet_pca_features         -> PCs (PC1..PCk) + chaves + label
#   2) emodnet_pca_metadata         -> k escolhido + variância explicada (por PC e acumulada)
#   3) emodnet_pca_loadings         -> loadings (matriz features x PCs)
#
# Observação metodológica:
# - O PCA é aplicado SOMENTE nas variáveis normalizadas (z-score), como no artigo.
# - Mantemos WQI e QualityClass fora do PCA (apenas carregadas como referência/label).

# PASSO 4 — PCA Multitemporal

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import PCA as SparkPCA
from pyspark.ml.functions import vector_to_array

# =========================
# CONFIG
# =========================
IN_TABLE = "emodnet_chemistry_integrated"

OUT_PCA_TABLE      = "emodnet_pca_features"
OUT_META_TABLE     = "emodnet_pca_metadata"
OUT_LOADINGS_TABLE = "emodnet_pca_loadings"

ZCOLS = ["PHO_z", "DO_z", "SIL_z", "DIN_z", "CHL_a_z"]
KEEP_COLS = ["time", "year", "latitude", "longitude", "WQI", "QualityClass"]

TARGET_CUMVAR = 0.95

# =========================
# LOAD
# =========================
df = spark.table(IN_TABLE)

# Remove nulos
df = df.dropna(subset=ZCOLS + ["QualityClass"])

print("Linhas disponíveis para PCA:", df.count())

# =========================
# 1️⃣ Vetorização
# =========================
assembler = VectorAssembler(inputCols=ZCOLS, outputCol="features_vec")
df_vec = assembler.transform(df).select(*(KEEP_COLS + ["features_vec"]))

# =========================
# 2️⃣ PCA completo para determinar k
# =========================
p = len(ZCOLS)

pca_full = SparkPCA(k=p, inputCol="features_vec", outputCol="pca_full_vec")
pca_full_model = pca_full.fit(df_vec)

expl_var = pca_full_model.explainedVariance.toArray().tolist()

cum = 0.0
k95 = 0
cum_list = []

for i, v in enumerate(expl_var, start=1):
    cum += float(v)
    cum_list.append(cum)
    if k95 == 0 and cum >= TARGET_CUMVAR:
        k95 = i

if k95 == 0:
    k95 = p

print("Explained variance:", [round(x, 6) for x in expl_var])
print("Cumulative variance:", [round(x, 6) for x in cum_list])
print(f"✅ k selecionado (≥95%): {k95}")

# =========================
# 3️⃣ PCA final
# =========================
pca = SparkPCA(k=k95, inputCol="features_vec", outputCol="pca_vec")
pca_model = pca.fit(df_vec)

df_pca = pca_model.transform(df_vec)

df_pca = df_pca.withColumn("pca_arr", vector_to_array("pca_vec"))

for i in range(k95):
    df_pca = df_pca.withColumn(f"PC{i+1}", F.col("pca_arr")[i].cast("double"))

pc_cols = [f"PC{i+1}" for i in range(k95)]

df_final = df_pca.select(*(KEEP_COLS + pc_cols))

# =========================
# 4️⃣ Salvar PCA features
# =========================
(
    df_final.write
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .format("delta")
      .saveAsTable(OUT_PCA_TABLE)
)

print(f"✅ PCA features salvas: {OUT_PCA_TABLE}")

# =========================
# 5️⃣ Salvar variância explicada
# =========================
expl_var_k = pca_model.explainedVariance.toArray().tolist()

meta_rows = []
cum2 = 0.0

for i, v in enumerate(expl_var_k, start=1):
    cum2 += float(v)
    meta_rows.append((
        i,
        float(v),
        float(cum2),
        int(k95),
        float(TARGET_CUMVAR)
    ))

meta_schema = T.StructType([
    T.StructField("pc_index", T.IntegerType(), False),
    T.StructField("explained_variance_ratio", T.DoubleType(), False),
    T.StructField("cumulative_explained_variance", T.DoubleType(), False),
    T.StructField("k_selected", T.IntegerType(), False),
    T.StructField("target_cumvar", T.DoubleType(), False),
])

meta_df = spark.createDataFrame(meta_rows, schema=meta_schema) \
               .withColumn("generated_at_utc", F.current_timestamp())

(
    meta_df.write
      .mode("overwrite")
      .format("delta")
      .saveAsTable(OUT_META_TABLE)
)

print(f"✅ Metadata salva: {OUT_META_TABLE}")

# =========================
# 6️⃣ Salvar loadings
# =========================
pc_matrix = pca_model.pc.toArray()

loadings_rows = []

for feat_i, feat_name in enumerate(ZCOLS):
    for pc_i in range(k95):
        loadings_rows.append((
            feat_name,
            int(pc_i + 1),
            float(pc_matrix[feat_i, pc_i])
        ))

loadings_schema = T.StructType([
    T.StructField("feature", T.StringType(), False),
    T.StructField("pc_index", T.IntegerType(), False),
    T.StructField("loading", T.DoubleType(), False),
])

loadings_df = spark.createDataFrame(loadings_rows, schema=loadings_schema) \
                   .withColumn("k_selected", F.lit(int(k95))) \
                   .withColumn("generated_at_utc", F.current_timestamp())

(
    loadings_df.write
      .mode("overwrite")
      .format("delta")
      .saveAsTable(OUT_LOADINGS_TABLE)
)

print(f"✅ Loadings salvos: {OUT_LOADINGS_TABLE}")

# Preview
spark.table(OUT_META_TABLE).show()



StatementMeta(, 63d61d78-39a8-45e0-808f-72e5ef490dc0, 3, Finished, Available, Finished, False)

Linhas disponíveis para PCA: 32480


Explained variance: [0.673111, 0.172532, 0.109284, 0.029365, 0.015708]
Cumulative variance: [0.673111, 0.845642, 0.954927, 0.984292, 1.0]
✅ k selecionado (≥95%): 3


✅ PCA features salvas: emodnet_pca_features
✅ Metadata salva: emodnet_pca_metadata
✅ Loadings salvos: emodnet_pca_loadings
+--------+------------------------+-----------------------------+----------+-------------+--------------------+
|pc_index|explained_variance_ratio|cumulative_explained_variance|k_selected|target_cumvar|    generated_at_utc|
+--------+------------------------+-----------------------------+----------+-------------+--------------------+
|       1|      0.6731105312173808|           0.6731105312173808|         3|         0.95|2026-03-31 12:33:...|
|       2|     0.17253182155437347|           0.8456423527717543|         3|         0.95|2026-03-31 12:33:...|
|       3|     0.10928441481170006|           0.9549267675834543|         3|         0.95|2026-03-31 12:33:...|
+--------+------------------------+-----------------------------+----------+-------------+--------------------+

